Mounted at /content/drive


In [ ]:
# %% [markdown]
# # Final Gated Strategy Backtest (v9 - The REAL Fix)
#
# This notebook implements the final, critical fix.
#
# 1.  **Data Prep:** Uses all our correct v8 data loaders.
# 2.  **Feature Engineering:** Builds all t-1 features (using a fresh cache).
# 3.  **WFO Engine (FIXED):**
#     * **Removes `.fillna(median())`**.
#     * Lets LightGBM handle NaNs natively as a signal.
# 4.  **Dynamic Discovery:** Programmatically finds the optimal rules for Puts and Calls.
# 5.  **Gating Logic:** Applies these *dynamically discovered* rules.
# 6.  **Global Guardrail:** Applies the final `L1_vvix_above_ema30` "kill switch".

# %% [code]
# =============================================================================
# SECTION 1: IMPORTS & SETUP
# =============================================================================
import pandas as pd
import numpy as np
import os
import re
import glob
import math
import datetime as dt
import warnings
from pathlib import Path

# Install LightGBM if not present
try:
    from lightgbm import LGBMRegressor
except ImportError:
    print("Installing lightgbm...")
    !pip install lightgbm -q
    from lightgbm import LGBMRegressor

# Install yfinance if not present
try:
    import yfinance as yf
except ImportError:
    print("Installing yfinance...")
    !pip install yfinance -q
    import yfinance as yf

# Suppress warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

# --- Paths ---
BASE_DIR = Path("/content/drive/MyDrive/xdte_selector/backtest_data")
if not BASE_DIR.exists():
    BASE_DIR = Path("/mnt/data") # Fallback
OUT_DIR = BASE_DIR / "_analysis_final_strategy" # Stable folder
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = BASE_DIR / "_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base Dir: {BASE_DIR}")
print(f"Output Dir: {OUT_DIR}")

# --- Strategy Config ---
BOOKS = ["PUTS_0DTE_11", "CALLS_0DTE_11", "PUTS_1DTE_1515", "CALLS_1DTE_1515"]
N_FOLDS = 8
ALPHA = 0.05 # 5th Percentile for Quantile Regression
KILL_SWITCH_EMA_LEN = 30 # Final optimized guardrail

# =============================================================================
# SECTION 2: HELPER FUNCTIONS (CORRECTED)
# =============================================================================

# --- Metrics ---
def _num(s):
    return pd.to_numeric(s, errors="coerce")

def _pf(arr):
    s = _num(arr).dropna()
    if s.empty: return np.nan
    g = s[s>0].sum(); l = -s[s<0].sum()
    if l==0: return (float("inf") if g>0 else np.nan)
    return float(g/l)

def _cvar(arr, alpha=0.05):
    s = np.asarray(_num(arr).dropna(), dtype=float)
    if s.size == 0: return np.nan
    k = max(1, int(np.floor(alpha * len(s))))
    idx = np.argsort(s)
    return float(s[idx[:k]].mean())

# --- WFO Splitter ---
def _wfo_splits(dates, n_folds=N_FOLDS):
    u = np.array(sorted(pd.to_datetime(pd.Series(dates).unique())))
    n = len(u)
    if n < max(30, n_folds + 1):
        chunks = np.array_split(u, min(n, n_folds + 1))
        out=[]
        for i in range(1, len(chunks)):
            tr = np.concatenate(chunks[:i])
            te = np.array(chunks[i])
            if len(tr)>=20 and len(te)>=5: out.append((tr, te))
        return out
    step = n // (n_folds + 1)
    out=[]
    for i in range(1, n_folds+1):
        cut = i*step
        tr = u[:cut]
        te = u[cut:min(cut+step, n)]
        if len(te)>=5: out.append((tr, te))
    return out

# --- Representative Trade Selector ---
def _rep_day(group, target=0.15):
    g = group.copy()
    d = None
    if "delta_abs" in g.columns and _num(g["delta_abs"]).notna().any():
        d = _num(g["delta_abs"]).abs()
    elif "declared_delta" in g.columns and _num(g["declared_delta"]).notna().any():
        d = _num(g["declared_delta"]).abs()

    if d is not None and d.notna().any():
        idx = (d - target).abs().idxmin()
        return g.loc[idx]
    if "credit" in g.columns and _num(g["credit"]).notna().any():
        return g.loc[_num(g["credit"]).idxmax()]
    return g.iloc[0]

# --- CORRECTED DATA LOADERS ---
DT_FORMATS = [
    "%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M",
    "%m/%d/%Y %H:%M:%S", "%m/%d/%Y %H:%M",
    "%Y-%m-%d %I:%M %p", "%m/%d/%Y %I:%M %p"
]
LEG_TOKEN = re.compile(r"\b(\d+(?:\.\d+)?)\s*([PCpc])\b")

def parse_dt(date_str, time_str):
    if not isinstance(date_str, str) or not isinstance(time_str, str): return pd.NaT
    s = f"{date_str.strip()} {time_str.strip()}"
    for fmt in DT_FORMATS:
        try: return pd.to_datetime(s, format=fmt)
        except Exception: continue
    return pd.to_datetime(s, errors="coerce")

def tag_session(ts: pd.Timestamp) -> str:
    if pd.isna(ts): return "off-schedule"
    t11   = ts.normalize() + pd.Timedelta(hours=11)
    t1515 = ts.normalize() + pd.Timedelta(hours=15, minutes=15)
    if abs((ts - t11).total_seconds())   <= 20*60: return "11:00"
    if abs((ts - t1515).total_seconds()) <= 20*60: return "15:15"
    return "off-schedule"

def make_book(session: str, side_pc: str) -> str:
    if not isinstance(side_pc, str): return "UNKNOWN"
    if session == "11:00":
        return "PUTS_0DTE_11" if side_pc=="P" else ("CALLS_0DTE_11" if side_pc=="C" else "UNKNOWN")
    if session == "15:15":
        return "PUTS_1DTE_1515" if side_pc=="P" else ("CALLS_1DTE_1515" if side_pc=="C" else "UNKNOWN")
    return "UNKNOWN"

def parse_legs(legs: str):
    if not isinstance(legs, str): return np.nan, np.nan, None
    m = LEG_TOKEN.findall(legs)
    if len(m) < 1: return np.nan, np.nan, None
    try:
        short_strike = float(m[0][0])
        side = m[0][1].upper()
        long_strike  = float(m[1][0]) if len(m) >= 2 else np.nan
        return short_strike, long_strike, side
    except Exception:
        return np.nan, np.nan, None

def _declared_delta_from_strategy(s: str):
    if not isinstance(s, str): return np.nan
    m = re.search(r'(\d+)\s*delta', s, flags=re.I)
    if not m: return np.nan
    try: return float(m.group(1)) / 100.0
    except: return np.nan

def load_portfolio(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False, encoding='utf-8', on_bad_lines='skip')
    df["file_source"] = os.path.basename(path)
    df["open_dt"]  = df.apply(lambda r: parse_dt(r.get("Date Opened"), r.get("Time Opened")), axis=1)
    df["close_dt"] = df.apply(lambda r: parse_dt(r.get("Date Closed"), r.get("Time Closed")), axis=1)

    s = df["Legs"].apply(parse_legs)
    df["short_strike"] = s.apply(lambda x: x[0])
    df["long_strike"]  = s.apply(lambda x: x[1])
    df["side_legs"]    = s.apply(lambda x: x[2])

    df["PNL"] = _num(df.get("P/L"))
    df["credit"] = _num(df.get("Premium"))

    if "Strategy" in df.columns:
        df["declared_delta"] = df["Strategy"].apply(_declared_delta_from_strategy)
    if "delta_abs" not in df.columns and "declared_delta" in df.columns:
        df["delta_abs"] = df["declared_delta"]

    df["session"] = df["open_dt"].apply(tag_session)
    wing_file = ("P" if "put" in os.path.basename(path).lower()
                   else ("C" if "call" in os.path.basename(path).lower() else ""))

    df["SIDE"] = df["side_legs"].fillna(wing_file).replace({"PUT":"P","CALL":"C"})
    df["book"] = df.apply(lambda r: make_book(r["session"], r["SIDE"]), axis=1)
    df["open_date"] = df["open_dt"].dt.normalize()
    return df

def load_market_stats(path: str) -> pd.DataFrame:
    g = pd.read_csv(path, low_memory=False, encoding='utf-8', on_bad_lines='skip')
    g.columns = [c.strip() for c in g.columns]
    base = os.path.basename(path).lower()

    rename_map = {}
    for c in g.columns:
        cl = c.lower().replace(" ", "_")
        if "opening_vix" in cl or ("vix" in cl and "entry" in cl):
            rename_map[c] = "VIX_Entry"
        if "intraday_move_opentoentry" in cl or "movement" in cl:
            rename_map[c] = "Intraday_Move_OpenToEntry"

    g = g.rename(columns=rename_map)

    missing = {"VIX_Entry", "Intraday_Move_OpenToEntry"} - set(g.columns)
    if missing:
        print(f"Warning: {base}: missing {missing}. Seen: {list(g.columns)}")

    date_col = next((c for c in g.columns if c.lower() in {"date","day","trade_date", "date_opened", "open_date"}), g.columns[0])
    g["mkt_date"] = pd.to_datetime(g[date_col], errors="coerce").dt.normalize()

    sess = "15:15" if "1515" in base else ("11:00" if "11" in base or "1100" in base else None)
    if not sess:
        print(f"Warning: {base}: cannot infer session from filename. Skipping file.")
        return pd.DataFrame()

    return pd.DataFrame({
        "mkt_date": g["mkt_date"],
        "session":  sess,
        "VIX_Entry": _num(g.get("VIX_Entry")),
        "Intraday_Move_OpenToEntry": _num(g.get("Intraday_Move_OpenToEntry"))
    }).dropna(subset=["mkt_date"])


# =============================================================================
# SECTION 3: DATA PREPARATION & FEATURE ENGINEERING
# =============================================================================

print("\n[Phase 1] Loading and Merging All Raw Data...")
portfolio_files = glob.glob(str(BASE_DIR / "portfolio_*.csv"))
trades = pd.concat([load_portfolio(p) for p in portfolio_files], ignore_index=True)
trades = trades[trades["book"].isin(BOOKS)].copy()
trades = trades.drop_duplicates(subset=["open_dt", "Legs", "file_source"]).reset_index(drop=True)
print(f"Loaded and filtered {len(trades)} trades from {len(portfolio_files)} files.")

mkt_files = glob.glob(str(BASE_DIR / "*market_stats*.csv"))
market = pd.concat([load_market_stats(p) for p in mkt_files], ignore_index=True)
market = market.drop_duplicates(subset=["mkt_date", "session"]).reset_index(drop=True)
print(f"Loaded {len(market)} market stat rows from {len(mkt_files)} files.")

pvt = market.pivot_table(index="mkt_date", columns="session",
                         values=["VIX_Entry","Intraday_Move_OpenToEntry"],
                         aggfunc="mean")
daily_t0 = pd.DataFrame(index=pvt.index)
daily_t0["VIX_Entry_11"] = pvt.get(('VIX_Entry', '11:00'), np.nan)
daily_t0["VIX_Entry_1515"] = pvt.get(('VIX_Entry', '15:15'), np.nan)
daily_t0["Intraday_Move_OpenToEntry_11"] = pvt.get(('Intraday_Move_OpenToEntry', '11:00'), np.nan)
daily_t0["Intraday_Move_OpenToEntry_1515"] = pvt.get(('Intraday_Move_OpenToEntry', '15:15'), np.nan)
daily_t0 = daily_t0.reset_index().rename(columns={"mkt_date":"open_date"})
daily_t0["open_date"] = pd.to_datetime(daily_t0["open_date"]).dt.normalize()

# --- Load Daily t-1 Context (yfinance) ---
print("\n[Phase 2] Engineering t-1 (Lagged) Features...")
yf_cache_file = CACHE_DIR / "daily_features_cache_v7.csv" # Force rebuild
TICKERS_TO_FETCH = {'GSPC': '^GSPC', 'VIX': '^VIX', 'VVIX': '^VVIX', 'VIX3M': '^VIX3M'}

if yf_cache_file.exists():
    print("Loading t-1 features from cache...")
    day = pd.read_csv(yf_cache_file, parse_dates=["open_date"])
else:
    print("Fetching t-1 features from yfinance...")
    try:
        end_date = dt.date.today()
        start_date = end_date - dt.timedelta(days=10*365)
        tkr_list = list(TICKERS_TO_FETCH.values())
        data = yf.download(tkr_list, start=start_date, end=end_date, auto_adjust=False, progress=False)
        day = pd.DataFrame(index=data.index[data.index.isin(data['Close']['^GSPC'].dropna().index)])
        day['open_date'] = day.index.normalize()

        def _get_col(df, tkr, col='Close'):
            if (col, tkr) in df.columns:
                return df[(col, tkr)]
            return pd.Series(index=df.index, name=f"{tkr}_{col}")

        day['SPX_Close'] = _get_col(data, '^GSPC', 'Close')
        spx_high = _get_col(data, '^GSPC', 'High')
        spx_low = _get_col(data, '^GSPC', 'Low')
        day['SPX_ATR_Pct'] = (spx_high - spx_low).ewm(span=14, adjust=False).mean() / day['SPX_Close']
        day['SPX_Drawdown_Pct'] = (day['SPX_Close'] - day['SPX_Close'].rolling(252, min_periods=20).max()) / day['SPX_Close'].rolling(252, min_periods=20).max()

        day['VIX_Close'] = _get_col(data, '^VIX', 'Close')
        day['VIX3M_Close'] = _get_col(data, '^VIX3M', 'Close')
        day['TS_eff'] = day['VIX_Close'] / day['VIX3M_Close']
        day['vix_pct'] = day['VIX_Close'].rank(pct=True)
        day['VIX_pct'] = day['vix_pct'] # Alias

        day['VVIX_Close'] = _get_col(data, '^VVIX', 'Close')
        day['vvix_ema20'] = day['VVIX_Close'].ewm(span=20, adjust=False).mean()
        day['vvix_ema30'] = day['VVIX_Close'].ewm(span=KILL_SWITCH_EMA_LEN, adjust=False).mean()
        day['vvix_pct'] = day['VVIX_Close'].rank(pct=True)

        ret = day['SPX_Close'].pct_change()
        day['rv5'] = ret.rolling(5).std() * (252**0.5)
        day['rv20'] = ret.rolling(20).std() * (252**0.5)
        day['DoW'] = day['open_date'].dt.weekday

        for col in day.columns:
            day[col] = day[col].reindex(day.index)
        day = day.reset_index(drop=True)
        day.to_csv(yf_cache_file, index=False)
        print("t-1 features saved to cache.")
    except Exception as e:
        print(f"CRITICAL ERROR: Could not download from yfinance. Error: {e}")
        day = pd.DataFrame()

# --- Lag all t-1 features ---
lag_cols = {
    'TS_eff': 'L1_TS', 'VIX_Close': 'L1_VIX_Close', 'vix_pct': 'L1_vix_pct',
    'VIX_pct': 'L1_VIX_pct', 'VVIX_Close': 'L1_vvix_close', 'vvix_pct': 'L1_vvix_pct',
    'vvix_ema20': 'L1_vvix_ema20', 'vvix_ema30': 'L1_vvix_ema30',
    'SPX_ATR_Pct': 'L1_SPX_ATR_Pct', 'SPX_Drawdown_Pct': 'L1_SPX_Drawdown_Pct',
    'rv5': 'L1_rv5', 'rv20': 'L1_rv20', 'DoW': 'DoW'
}
for src, tgt in lag_cols.items():
    if src in day.columns:
        day[tgt] = day[src].shift(1) if src != 'DoW' else day[src]
    else:
        print(f"Warning: Source col {src} not in yf data. Cannot create {tgt}.")

# --- Engineer Kill Switch & t-0 Melt-up Features ---
if 'L1_vvix_close' in day.columns and 'L1_vvix_ema30' in day.columns:
    day['L1_vvix_above_ema30'] = (day['L1_vvix_close'] > day['L1_vvix_ema30'])
if 'L1_vvix_close' in day.columns and 'L1_vvix_ema20' in day.columns:
    day['L1_vvix_above_ema20'] = (day['L1_vvix_close'] > day['L1_vvix_ema20'])

print("\n[Phase 3] Creating Final Master Panel...")
rep_rows = []
for (book, d), g in trades.groupby(["book","open_date"], sort=True):
    rep_rows.append(_rep_day(g))
rep = pd.DataFrame(rep_rows).reset_index(drop=True)
pday = rep.groupby("open_date", as_index=False).agg(portfolio_pnl=("PNL","sum"))
book_panel = rep.merge(daily_t0, on="open_date", how="left")
book_panel = book_panel.merge(day, on="open_date", how="left")
if 'VIX_Entry_11' in book_panel.columns and 'L1_VIX_Close' in book_panel.columns:
    book_panel['t0_VIX_change_from_close_11'] = book_panel['VIX_Entry_11'] - book_panel['L1_VIX_Close']
if 'VIX_Entry_1515' in book_panel.columns and 'L1_VIX_Close' in book_panel.columns:
    book_panel['t0_VIX_change_from_close_15'] = book_panel['VIX_Entry_1515'] - book_panel['L1_VIX_Close']

print(f"Master panel created with {len(book_panel)} rows.")
print("\n[QC] Per-book sample counts:")
print(rep['book'].value_counts().to_string())
print("\n[QC] t=0 feature NaN rates (should be low):")
for col in ['VIX_Entry_11','Intraday_Move_OpenToEntry_11','VIX_Entry_1515','Intraday_Move_OpenToEntry_1515']:
    if col in book_panel.columns:
        relevant_books = [b for b in BOOKS if col.split("_")[-1] in b]
        nan_rate = book_panel[book_panel['book'].isin(relevant_books)][col].isna().mean()
        print(f"{col}: {nan_rate:.2%}")


Base Dir: /content/drive/MyDrive/xdte_selector/backtest_data
Output Dir: /content/drive/MyDrive/xdte_selector/backtest_data/_analysis_final_strategy

[Phase 1] Loading and Merging All Raw Data...
Loaded and filtered 36649 trades from 7 files.
Loaded 1494 market stat rows from 2 files.

[Phase 2] Engineering t-1 (Lagged) Features...
Loading t-1 features from cache...

[Phase 3] Creating Final Master Panel...
Master panel created with 2985 rows.

[QC] Per-book sample counts:
book
CALLS_0DTE_11      750
PUTS_0DTE_11       750
PUTS_1DTE_1515     743
CALLS_1DTE_1515    742

[QC] t=0 feature NaN rates (should be low):
VIX_Entry_11: 0.00%
Intraday_Move_OpenToEntry_11: 0.00%
VIX_Entry_1515: 0.00%
Intraday_Move_OpenToEntry_1515: 0.00%


In [ ]:
# === Cell 0: Feature config used by Cell A ===
BOOK_FEATS_CONFIG = {
    "PUTS_0DTE_11": [
        "Intraday_Move_OpenToEntry_11","VIX_Entry_11",
        "L1_TS","L1_vvix_pct","L1_SPX_ATR_Pct","L1_SPX_Drawdown_Pct","DoW",
        "L1_rv20","L1_vvix_above_ema20"
    ],
    "PUTS_1DTE_1515": [
        "Intraday_Move_OpenToEntry_1515","VIX_Entry_1515","t0_VIX_change_from_close_15",
        "L1_TS","L1_vvix_pct","L1_SPX_ATR_Pct","L1_SPX_Drawdown_Pct","DoW",
        "L1_rv20","L1_vvix_above_ema20"
    ],
    "CALLS_0DTE_11": [
        "Intraday_Move_OpenToEntry_11","VIX_Entry_11","t0_VIX_change_from_close_11",
        "L1_TS","L1_SPX_ATR_Pct","L1_VIX_pct","DoW"
    ],
    "CALLS_1DTE_1515": [
        "Intraday_Move_OpenToEntry_1515","VIX_Entry_1515","t0_VIX_change_from_close_15",
        "L1_TS","L1_SPX_ATR_Pct","L1_VIX_pct","DoW"
    ],
}


In [ ]:
# =============== CELL A (FIXED): OOS prediction (fold artifacts) ===============
import numpy as np, pandas as pd
from lightgbm import LGBMRegressor

np.random.seed(42)

# LightGBM helper (FIXED)
def _fit_lgbm_quantile(X_tr, y_tr, alpha=0.05, seed=42):
    return LGBMRegressor(
        objective='quantile', alpha=alpha,
        n_estimators=100, random_state=seed, n_jobs=-1, verbose=-1
    ).fit(X_tr, y_tr)

def _wfo_splits(dates, n_folds):
    u = np.array(sorted(pd.to_datetime(pd.Series(dates).unique())))
    n = len(u)
    step = max(1, n // (n_folds + 1)) if (n_folds + 1) > 0 else n
    out=[]
    for i in range(1, n_folds+1):
        cut = i*step
        tr = u[:cut]; te = u[cut:min(cut+step, n)]
        if len(te) >= 5 and len(tr) >= 20:
            out.append((tr, te))
    if not out:
        chunks = np.array_split(u, min(n, n_folds+1))
        for i in range(1, len(chunks)):
            tr = np.concatenate(chunks[:i]); te = np.array(chunks[i])
            if len(te) >= 5 and len(tr) >= 20:
                out.append((tr, te))
    return out

def _coerce_numeric(df, cols):
    """Return numeric-only DataFrame for cols: coerce to numeric, drop fully-NaN & constant."""
    z = df[cols].copy()
    for c in z.columns:
        if not (pd.api.types.is_numeric_dtype(z[c]) or pd.api.types.is_bool_dtype(z[c])):
            z[c] = pd.to_numeric(z[c], errors='coerce')
    keep = [c for c in z.columns if (z[c].notna().sum()>0 and z[c].nunique(dropna=True)>1)]
    return z[keep], [c for c in cols if c not in keep]

# columns we never want to model on (even if numeric after coercion)
_NON_FEATURES = {"open_date","book","PNL","kill_switch"}

print("\n[Cell A] Generating OOS predictions and artifacts (numeric features only)...")

fold_artifacts = {b: [] for b in BOOKS}
oos_rows = []

for book in BOOKS:
    df = book_panel[book_panel['book']==book].copy().sort_values('open_date')
    if df.empty:
        print(f"[WARN] {book}: no rows")
        continue

    # Use configured feature set; if missing, fallback to auto-select numeric (minus non-features)
    if book in BOOK_FEATS_CONFIG:
        raw_feats = [f for f in BOOK_FEATS_CONFIG[book] if f in df.columns]
    else:
        raw_feats = [c for c in df.columns if c not in _NON_FEATURES]

    if not raw_feats:
        print(f"[WARN] {book}: no configured features present.")
        continue

    # Coerce to numeric and drop non-informative
    X_all, dropped = _coerce_numeric(df, raw_feats)
    feats = list(X_all.columns)

    # Audit
    if dropped:
        print(f"[AUDIT] {book}: dropped non-numeric/constant: {dropped}")
    if feats:
        worst = (X_all[feats].isna().mean()*100).round(1).sort_values(ascending=False).head(5).to_dict()
        print(f"[AUDIT] {book}: NaN% (worst 5): {worst}")

    if not feats:
        print(f"[WARN] {book}: no usable numeric features after coercion.")
        continue

    alpha = ALPHA  # same alpha for all unless you intentionally change

    for tr_days, te_days in _wfo_splits(df["open_date"].values, N_FOLDS):
        tr_idx = df["open_date"].isin(tr_days)
        te_idx = df["open_date"].isin(te_days)
        tr = df.loc[tr_idx]; te = df.loc[te_idx]
        if tr.empty or te.empty:
            continue

        X_tr = X_all.loc[tr.index, feats]
        X_te = X_all.loc[te.index, feats]
        y_tr = tr["PNL"].fillna(0)
        y_te = te["PNL"]

        m = _fit_lgbm_quantile(X_tr, y_tr, alpha=alpha, seed=42)
        s_tr = pd.Series(m.predict(X_tr), index=tr.index)
        s_te = pd.Series(m.predict(X_te), index=te.index)

        fold_artifacts[book].append({
            "train_scores": s_tr,
            "test_scores":  s_te,
            "train_pnl":    tr["PNL"].copy(),
            "test_pnl":     y_te.copy(),
            "test_dates":   te["open_date"].copy()
        })

        oos_rows.append(pd.DataFrame({
            "open_date": te["open_date"].values,
            "book":      book,
            "PNL":       y_te.values,
            "score":     s_te.values
        }))

oos_panel = pd.concat(oos_rows, ignore_index=True) if oos_rows else pd.DataFrame(columns=["open_date","book","PNL","score"])
print(f"[A] OOS rows: {len(oos_panel)}; folds per book:", {b: len(fold_artifacts[b]) for b in BOOKS})




[Cell A] Generating OOS predictions and artifacts (numeric features only)...
[AUDIT] PUTS_0DTE_11: NaN% (worst 5): {'Intraday_Move_OpenToEntry_11': 0.0, 'VIX_Entry_11': 0.0, 'L1_TS': 0.0, 'L1_vvix_pct': 0.0, 'L1_SPX_ATR_Pct': 0.0}
[AUDIT] CALLS_0DTE_11: NaN% (worst 5): {'Intraday_Move_OpenToEntry_11': 0.0, 'VIX_Entry_11': 0.0, 't0_VIX_change_from_close_11': 0.0, 'L1_TS': 0.0, 'L1_SPX_ATR_Pct': 0.0}
[AUDIT] PUTS_1DTE_1515: NaN% (worst 5): {'Intraday_Move_OpenToEntry_1515': 0.0, 'VIX_Entry_1515': 0.0, 't0_VIX_change_from_close_15': 0.0, 'L1_TS': 0.0, 'L1_vvix_pct': 0.0}
[AUDIT] CALLS_1DTE_1515: NaN% (worst 5): {'Intraday_Move_OpenToEntry_1515': 0.0, 'VIX_Entry_1515': 0.0, 't0_VIX_change_from_close_15': 0.0, 'L1_TS': 0.0, 'L1_SPX_ATR_Pct': 0.0}
[A] OOS rows: 2640; folds per book: {'PUTS_0DTE_11': 8, 'CALLS_0DTE_11': 8, 'PUTS_1DTE_1515': 8, 'CALLS_1DTE_1515': 8}


In [ ]:
# =============== CELL B: Rule discovery (winsorized) ==========================
import numpy as np, pandas as pd, json

PUT_KEEP_GRID        = np.round(np.linspace(0.60, 0.90, 7), 2)  # proven band
MIN_DECILE_DAYS      = 20
WINSOR_P             = 0.01
PUT_CVAR_IMPROVE_FRAC= 0.10
CALLS_TOPK_LONG      = 2
CALLS_TOPK_SHORT     = 1

def _winsor(s, p=WINSOR_P):
    if len(s)==0 or pd.Series(s).isna().all(): return pd.Series(s)
    lo, hi = np.nanpercentile(s, [100*p, 100*(1-p)])
    return pd.Series(s).clip(lo, hi)

def _pf(s):
    s = pd.Series(s).dropna(); g=s[s>0].sum(); l=-s[s<0].sum()
    return float(g/l) if l>0 else (float("inf") if g>0 else np.nan)

def _cvar(s, alpha=0.05):
    v = pd.Series(s).dropna().values
    if v.size==0: return np.nan
    k=max(1,int(np.floor(alpha*len(v)))); idx=np.argsort(v)
    return float(np.mean(v[idx[:k]]))

DISCOVERED_RULES = {}

# PUTS: keep% maximizing CVaR improvement (winsorized selection)
for book in [b for b in BOOKS if b.startswith("PUTS")]:
    df = oos_panel[oos_panel['book']==book].copy()
    base = _winsor(df['PNL'])
    base_cvar = _cvar(base)
    target = base_cvar + abs(base_cvar)*PUT_CVAR_IMPROVE_FRAC
    rows=[]
    for keep in PUT_KEEP_GRID:
        tau = np.nanquantile(df['score'].values, 1.0-keep)
        kept = _winsor(df.loc[df['score']>=tau, 'PNL'])
        rows.append({"keep": float(keep), "EDP": kept.mean(), "PF": _pf(kept), "CVaR95": _cvar(kept)})
    dfc = pd.DataFrame(rows)
    feas = dfc[dfc["CVaR95"] >= target]
    choice = (feas if not feas.empty else dfc).sort_values(["CVaR95","EDP","PF"], ascending=[False,False,False]).iloc[0]
    DISCOVERED_RULES[book] = {"type":"filter","keep_target": float(choice["keep"])}
# CALLS: cross-fold OOS deciles, top-K per side (robust)
def _edges_from_scores(arr, n=10):
    arr = pd.Series(arr).astype(float)
    if arr.notna().sum() < 5:
        return None  # too few
    edges = np.unique(np.nanpercentile(arr.dropna().values, np.linspace(0, 100, n+1)))
    if len(edges) < 3:
        edges = np.unique(np.nanpercentile(arr.dropna().values, np.linspace(0, 100, 6)))
    return edges if len(edges) >= 3 else None

def _safe_bin(scores, edges):
    # returns integer decile labels or NaN if edges invalid
    if edges is None:
        return pd.Series(np.nan, index=pd.RangeIndex(len(scores)))
    return pd.Series(pd.cut(scores, edges, labels=False, include_lowest=True))

for book in [b for b in BOOKS if b.startswith("CALLS")]:
    fold_tabs = []
    folds = fold_artifacts.get(book, [])
    if not folds:
        print(f"[WARN] {book}: no folds -> skipping discovery for this book")
        continue

    for fa in folds:
        s_te = pd.Series(fa["test_scores"].values)
        p_te = pd.Series(fa["test_pnl"].values)
        # Build edges from TRAIN (safer, fold-calibrated)
        s_tr = pd.Series(fa["train_scores"].values)
        edges = _edges_from_scores(s_tr.values, n=10)
        dec = _safe_bin(s_te.values, edges)
        df_fold = pd.DataFrame({"dec": dec, "PNL": p_te})
        # keep only rows where dec successfully assigned
        df_fold = df_fold.dropna(subset=["dec"])
        if not df_fold.empty:
            fold_tabs.append(df_fold)

    if not fold_tabs:
        print(f"[WARN] {book}: all folds were degenerate (flat/NaN scores) -> default to skip")
        DISCOVERED_RULES[book] = {"type":"sweep", "long_deciles": [], "short_deciles": []}
        continue

    pooled = pd.concat(fold_tabs, ignore_index=True)
    pooled["PNL_w"] = _winsor(pooled["PNL"])
    tab = (pooled.groupby("dec")["PNL_w"]
                 .agg(N="count", Short_EDP="mean", Long_EDP=lambda s: (-s).mean())
                 .reset_index())

    # require minimum days per decile
    tab = tab[tab["N"] >= MIN_DECILE_DAYS]
    if tab.empty:
        print(f"[WARN] {book}: no deciles met MIN_DECILE_DAYS={MIN_DECILE_DAYS} -> default to skip")
        DISCOVERED_RULES[book] = {"type":"sweep", "long_deciles": [], "short_deciles": []}
        continue

    longs  = (tab.sort_values("Long_EDP",  ascending=False)
                .head(CALLS_TOPK_LONG)["dec"].astype(int).tolist())
    shorts = (tab.sort_values("Short_EDP", ascending=False)
                .head(CALLS_TOPK_SHORT)["dec"].astype(int).tolist())

    DISCOVERED_RULES[book] = {"type":"sweep", "long_deciles": longs, "short_deciles": shorts}
    print(f"[CALLS DISCOVERY] {book}: long={longs} short={shorts}")


[CALLS DISCOVERY] CALLS_0DTE_11: long=[0, 3] short=[2]
[CALLS DISCOVERY] CALLS_1DTE_1515: long=[2, 3] short=[0]


In [ ]:
# === Cell C PATCH: fold-calibrated apply with CVaR-based side selection for PUTS ===
import numpy as np, pandas as pd, json

with open(OUT_DIR / "discovered_rules.json") as f:
    RULES = json.load(f)

def _pf(s):
    s = pd.Series(s).dropna(); g=s[s>0].sum(); l=-s[s<0].sum()
    return float(g/l) if l>0 else (float("inf") if g>0 else np.nan)

def _cvar(s, alpha=0.05):
    v = pd.Series(s).dropna().values
    if v.size==0: return np.nan
    k=max(1,int(np.floor(alpha*len(v)))); idx=np.argsort(v)
    return float(np.mean(v[idx[:k]]))

def _train_edges(scores, n=10):
    e = np.unique(np.nanpercentile(scores, np.linspace(0,100,n+1)))
    return e if len(e) >= 3 else np.unique(np.nanpercentile(scores, np.linspace(0,100,6)))

def apply_rules_foldcal(rules):
    days = np.sort(book_panel['open_date'].unique())
    out = pd.DataFrame(index=days)

    for book in BOOKS:
        chunks=[]

        if rules.get(book,{}).get("type")=="filter":  # PUTS
            keep = float(rules[book]["keep_target"])

            for fa in fold_artifacts.get(book, []):
                s_tr = pd.Series(fa["train_scores"].values, index=fa["train_pnl"].index)
                s_te = pd.Series(fa["test_scores"].values,  index=fa["test_pnl"].index)
                p_tr = pd.Series(fa["train_pnl"].values,    index=fa["train_pnl"].index)
                p_te = pd.Series(fa["test_pnl"].values,     index=fa["test_pnl"].index)

                if keep >= 0.999:
                    g = p_te.copy()
                else:
                    # τ from TRAIN for both sides
                    tau_top = np.nanquantile(s_tr.values, 1.0 - keep)
                    tau_bot = np.nanquantile(s_tr.values, keep)

                    # TRAIN masks
                    m_top_tr = (s_tr >= tau_top)
                    m_bot_tr = (s_tr <= tau_bot)

                    # TRAIN CVaR for both sides
                    cvar_top = _cvar(p_tr.where(m_top_tr, 0.0))
                    cvar_bot = _cvar(p_tr.where(m_bot_tr, 0.0))

                    # Choose side that maximizes TRAIN CVaR (less negative)
                    use_top = (pd.notna(cvar_top) and pd.notna(cvar_bot) and (cvar_top >= cvar_bot))
                    if pd.isna(cvar_top) and pd.notna(cvar_bot): use_top = False
                    if pd.notna(cvar_top) and pd.isna(cvar_bot): use_top = True

                    if use_top:
                        m_te = (s_te >= tau_top)
                    else:
                        m_te = (s_te <= tau_bot)

                    g = p_te.where(m_te, 0.0)

                g.index = fa["test_dates"].values
                chunks.append(g)

            # pooled-OOS fallback if EDP <= 0
            if chunks:
                ser = pd.concat(chunks).groupby(level=0).sum().reindex(days).fillna(0.0)
                if ser.mean() <= 0:
                    df_oos = oos_panel[oos_panel['book']==book][['open_date','PNL','score']].copy()
                    if not df_oos.empty:
                        # evaluate both sides on pooled OOS and pick best CVaR
                        tau_top = np.nanquantile(df_oos['score'].values, 1.0 - keep)
                        tau_bot = np.nanquantile(df_oos['score'].values, keep)
                        m_top = (df_oos['score'] >= tau_top)
                        m_bot = (df_oos['score'] <= tau_bot)
                        c_top = _cvar(df_oos.loc[m_top,'PNL'])
                        c_bot = _cvar(df_oos.loc[m_bot,'PNL'])
                        use_top = (pd.notna(c_top) and pd.notna(c_bot) and (c_top >= c_bot))
                        if pd.isna(c_top) and pd.notna(c_bot): use_top = False
                        if pd.notna(c_top) and pd.isna(c_bot): use_top = True
                        kept = df_oos.loc[m_top if use_top else m_bot].groupby('open_date')['PNL'].sum()
                        ser = kept.reindex(days).fillna(0.0)
            else:
                ser = pd.Series(0.0, index=days)

            out[book] = ser

        elif rules.get(book,{}).get("type")=="sweep":  # CALLS
            longs  = set(int(d) for d in rules[book]["long_deciles"])
            shorts = set(int(d) for d in rules[book]["short_deciles"])

            for fa in fold_artifacts.get(book, []):
                s_tr = pd.Series(fa["train_scores"].values, index=fa["train_pnl"].index)
                s_te = pd.Series(fa["test_scores"].values,  index=fa["test_pnl"].index)
                p_te = pd.Series(fa["test_pnl"].values,     index=fa["test_pnl"].index)

                edges = _train_edges(s_tr.values, n=10)
                dec = pd.Series(pd.cut(s_te.values, edges, labels=False, include_lowest=True), index=s_te.index)

                g = (-p_te).where(dec.isin(longs), 0.0) + ( p_te).where(dec.isin(shorts), 0.0)
                g.index = fa["test_dates"].values
                chunks.append(g)

            out[book] = (pd.concat(chunks).groupby(level=0).sum().reindex(days).fillna(0.0)) if chunks else pd.Series(0.0, index=days)

        else:
            out[book] = pd.Series(0.0, index=days)

    out["Gated_Portfolio_Base"] = out[BOOKS].sum(axis=1)
    return out

# re-apply with the CVaR-side selection
final_daily_pnl = apply_rules_foldcal(RULES)
print("[PATCH] Per-book EDP (no-kill) after CVaR-side selection:")
print(final_daily_pnl[BOOKS].mean().round(2).to_string())
print("[PATCH] Portfolio EDP (no-kill):", round(final_daily_pnl["Gated_Portfolio_Base"].mean(),2))


[PATCH] Per-book EDP (no-kill) after CVaR-side selection:
PUTS_0DTE_11       24.94
CALLS_0DTE_11      12.97
PUTS_1DTE_1515     53.45
CALLS_1DTE_1515    85.38
[PATCH] Portfolio EDP (no-kill): 176.73


In [ ]:
# =============== CELL D: Hybrid risk controls & report =======================
import numpy as np, pandas as pd, json

# Kill mask aligned to days (NO writing into final_daily_pnl yet)
DAYS = final_daily_pnl.index
ks = (day.set_index("open_date")["L1_vvix_above_ema30"]
        .reindex(DAYS).fillna(False).astype(bool))

# Load tuned hybrid policy or use reasonable defaults (you can swap in tuner outputs)
HYBRID_JSON = OUT_DIR / "hybrid_policy_best.json"
try:
    with open(HYBRID_JSON) as f:
        POLICY = json.load(f)
except Exception:
    POLICY = {
        "tails": { "PUTS_0DTE_11": [0.045, 0.961], "CALLS_0DTE_11": [0.0345, 0.952] },
        "gammas": { "PUTS_1DTE_1515": 1.0, "CALLS_1DTE_1515": 1.0 }
    }

def _apply_tail_on_kill(book, ql, qh):
    arts = fold_artifacts.get(book, [])
    base = final_daily_pnl[book].astype(float)
    if not arts: return base
    parts=[]
    for fa in arts:
        s_tr = fa["train_scores"].values
        s_te = pd.Series(fa["test_scores"].values, index=fa["test_pnl"].index)
        p_te = fa["test_pnl"].copy(); dts = fa["test_dates"].values
        tau_l = np.nanpercentile(s_tr, ql*100); tau_h = np.nanpercentile(s_tr, qh*100)
        pnl = (-p_te).where(s_te<=tau_l, 0.0) + (p_te).where(s_te>=tau_h, 0.0)
        pnl.index = dts; parts.append(pnl)
    tails = pd.concat(parts).groupby(level=0).sum().reindex(DAYS).fillna(0.0)
    return base.where(~ks, tails)

def _apply_gamma_on_kill(book, g):
    s = final_daily_pnl[book].astype(float)
    return s.where(~ks, s*float(g))

# Build hybrid production
prod = pd.DataFrame(index=DAYS)
for b in BOOKS:
    if b in POLICY.get("tails", {}):
        ql, qh = POLICY["tails"][b]
        prod[b] = _apply_tail_on_kill(b, ql, qh)
    elif b in POLICY.get("gammas", {}):
        prod[b] = _apply_gamma_on_kill(b, POLICY["gammas"][b])
    else:
        prod[b] = final_daily_pnl[b].astype(float)

# Reports
no_kill = final_daily_pnl["Gated_Portfolio_Base"]
hybrid  = prod[BOOKS].sum(axis=1)

def _pf(s):
    s = pd.Series(s).dropna(); g=s[s>0].sum(); l=-s[s<0].sum()
    return float(g/l) if l>0 else (float("inf") if g>0 else np.nan)
def _cvar(s, alpha=0.05):
    v=pd.Series(s).dropna().values
    if v.size==0: return np.nan
    k=max(1,int(np.floor(alpha*len(v)))); idx=np.argsort(v)
    return float(np.mean(v[idx[:k]]))

summary = pd.DataFrame({
    "EDP":    [no_kill.mean(), hybrid.mean()],
    "PF":     [_pf(no_kill),   _pf(hybrid)],
    "CVaR95": [_cvar(no_kill), _cvar(hybrid)],
    "Keep %": [1.00, 1.00]
}, index=["No Kill (rules)", "Hybrid Policy (prod)"])
print("\n[D] Hybrid summary:")
print(summary.to_string(float_format=lambda x: f"{x:,.2f}"))

# QC: tail participation
for b, q in POLICY.get("tails", {}).items():
    rep = (prod[b][ks] != 0).mean() if ks.any() else 0.0
    print(f"[QC] {b}: traded on {rep:.1%} of kill days using tails {tuple(round(x,3) for x in q)}")

# Save daily series
prod_out = OUT_DIR / "portfolio_series_hybrid.csv"
pd.DataFrame({"no_kill": no_kill, "hybrid": hybrid}).to_csv(prod_out)
print(f"[D] Saved daily portfolio series → {prod_out.name}")



[D] Hybrid summary:
                        EDP   PF    CVaR95  Keep %
No Kill (rules)      176.73 1.59 -4,057.59    1.00
Hybrid Policy (prod) 173.52 1.88 -2,894.76    1.00
[QC] PUTS_0DTE_11: traded on 18.3% of kill days using tails (0.061, 0.961)
[QC] CALLS_0DTE_11: traded on 4.7% of kill days using tails (0.051, 0.949)
[D] Saved daily portfolio series → portfolio_series_hybrid.csv


In [ ]:
# =============== CELL E: Hybrid tuner (optional) ==============================
# Only run if you want to re-tune tails/gammas; otherwise skip this cell.
import sys, subprocess, json, numpy as np, pandas as pd

try:
    import optuna
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)
    import optuna

ks = (day.set_index("open_date")["L1_vvix_above_ema30"].reindex(DAYS).fillna(False).astype(bool))

BOOKS_TAIL  = [b for b in ["PUTS_0DTE_11","CALLS_0DTE_11"] if b in BOOKS]
BOOKS_GAMMA = [b for b in ["PUTS_1DTE_1515","CALLS_1DTE_1515"] if b in BOOKS]

def _apply_tail(book, ql, qh):
    return _apply_tail_on_kill(book, ql, qh)
def _apply_gamma(book, g):
    return _apply_gamma_on_kill(book, g)

base = final_daily_pnl["Gated_Portfolio_Base"]
EDP_FLOOR = 0.95*base.mean()   # keep ≥95% EDP
CVaR_FLOOR = _cvar(base)       # no worse CVaR

def eval_policy(policy):
    comp=[]
    for b in BOOKS:
        if b in policy["tails"]:
            ql,qh = policy["tails"][b]; comp.append(_apply_tail(b, ql, qh))
        elif b in policy["gammas"]:
            comp.append(_apply_gamma(b, policy["gammas"][b]))
        else:
            comp.append(final_daily_pnl[b].astype(float))
    port = pd.concat(comp, axis=1).sum(axis=1)
    return {"EDP": port.mean(), "PF": _pf(port), "CVaR95": _cvar(port)}

def loss(m):
    penalty = 0.0
    if m["EDP"] < EDP_FLOOR:  penalty += 1000*(EDP_FLOOR - m["EDP"])
    if m["CVaR95"] < CVaR_FLOOR: penalty += 0.5*(CVaR_FLOOR - m["CVaR95"])
    return penalty - m["PF"] - 0.0005*m["EDP"]

def objective(trial):
    tails = {}
    for b in BOOKS_TAIL:
        ql = trial.suggest_float(f"{b}_ql", 0.01, 0.10)
        qh = trial.suggest_float(f"{b}_qh", 0.90, 0.99)
        if ql>=qh: ql, qh = 0.05, 0.95
        tails[b]=(ql,qh)
    gammas = {}
    for b in BOOKS_GAMMA:
        gammas[b] = trial.suggest_categorical(f"{b}_g", [1.0, 0.75, 0.5, 0.25, 0.0])
    pol = {"tails": tails, "gammas": gammas}
    return loss(eval_policy(pol))

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=80, show_progress_bar=False)

best = study.best_params
best_policy = {"tails": {}, "gammas": {}}
for b in BOOKS_TAIL:  best_policy["tails"][b]=(best[f"{b}_ql"], best[f"{b}_qh"])
for b in BOOKS_GAMMA: best_policy["gammas"][b]= float(best[f"{b}_g"])

best_metrics = eval_policy(best_policy)
print("\n[E] Tuner best:", best_metrics)
print(json.dumps(best_policy, indent=2))

# Save if better than current hybrid on PF & CVaR and meets EDP floor
cur_metrics = eval_policy(POLICY) if 'POLICY' in globals() else {"EDP":-1,"PF":-1,"CVaR95":-9e9}
if (best_metrics["EDP"]>=EDP_FLOOR and
    best_metrics["PF"] >= cur_metrics["PF"] and
    best_metrics["CVaR95"] >= cur_metrics["CVaR95"]):
    with open(OUT_DIR / "hybrid_policy_best.json","w") as f: json.dump(best_policy, f, indent=2)
    print("[E] Updated hybrid_policy_best.json")
else:
    print("[E] Kept existing hybrid policy")


[I 2025-11-09 22:37:53,135] A new study created in memory with name: no-name-54eed01d-4c32-4088-9544-14a622dae68e
[I 2025-11-09 22:37:53,167] Trial 0 finished with value: 20595.892702645717 and parameters: {'PUTS_0DTE_11_ql': 0.04370861069626263, 'PUTS_0DTE_11_qh': 0.9855642875768924, 'CALLS_0DTE_11_ql': 0.07587945476302646, 'CALLS_0DTE_11_qh': 0.9538792635777333, 'PUTS_1DTE_1515_g': 0.25, 'CALLS_1DTE_1515_g': 0.5}. Best is trial 0 with value: 20595.892702645717.
[I 2025-11-09 22:37:53,197] Trial 1 finished with value: 35373.34589777838 and parameters: {'PUTS_0DTE_11_ql': 0.02636424704863906, 'PUTS_0DTE_11_qh': 0.9165064058868091, 'CALLS_0DTE_11_ql': 0.0373818018663584, 'CALLS_0DTE_11_qh': 0.9472280788469014, 'PUTS_1DTE_1515_g': 0.5, 'CALLS_1DTE_1515_g': 0.5}. Best is trial 0 with value: 20595.892702645717.
[I 2025-11-09 22:37:53,229] Trial 2 finished with value: 1819.6235727273743 and parameters: {'PUTS_0DTE_11_ql': 0.06331731119758383, 'PUTS_0DTE_11_qh': 0.9041805371447998, 'CALLS_0D


[E] Tuner best: {'EDP': np.float64(174.21171770972038), 'PF': 1.8761862028783627, 'CVaR95': -2878.0}
{
  "tails": {
    "PUTS_0DTE_11": [
      0.06068177867703731,
      0.9081364486968552
    ],
    "CALLS_0DTE_11": [
      0.04501090308456395,
      0.9398913897324598
    ]
  },
  "gammas": {
    "PUTS_1DTE_1515": 0.0,
    "CALLS_1DTE_1515": 1.0
  }
}
[E] Kept existing hybrid policy


In [ ]:
# =============== CELL F: Persist + final report ==============================
(final_daily_pnl[["Gated_Portfolio_Base"] + BOOKS]
 .to_csv(OUT_DIR / "final_daily_pnl_nokill.csv"))

with open(OUT_DIR / "discovered_rules.json") as f:
    rules_txt = f.read()
with open(OUT_DIR / "run_rules_snapshot.json","w") as f:
    f.write(rules_txt)

print("\n[F] Artifacts written:")
print(" - discovered_rules.json (active rules)")
print(" - run_rules_snapshot.json (snapshot)")
print(" - final_daily_pnl_nokill.csv (per-day PnL)")

# Summary printout
nk = final_daily_pnl["Gated_Portfolio_Base"]
print("\n[F] No-kill summary:",
      {"EDP": nk.mean(), "PF": _pf(nk), "CVaR95": _cvar(nk)})
# === Optional: print the latest hybrid policy summary ===
HYBRID_JSON = OUT_DIR / "hybrid_policy_best.json"
try:
    with open(HYBRID_JSON) as f:
        policy = json.load(f)
        print("\n[F] Active hybrid policy parameters:")
        print(json.dumps(policy, indent=2))
except FileNotFoundError:
    print("\n[F] No hybrid policy file found; tuner has not yet produced one.")

# Evaluate and display hybrid policy metrics
hybrid = pd.read_csv(OUT_DIR / "portfolio_series_hybrid.csv")
hybrid_summary = {
    "EDP": hybrid["hybrid"].mean(),
    "PF":  _pf(hybrid["hybrid"]),
    "CVaR95": _cvar(hybrid["hybrid"])
}
print("\n[F] Hybrid portfolio summary:", {k: round(v,2) for k,v in hybrid_summary.items()})




[F] Artifacts written:
 - discovered_rules.json (active rules)
 - run_rules_snapshot.json (snapshot)
 - final_daily_pnl_nokill.csv (per-day PnL)

[F] No-kill summary: {'EDP': np.float64(176.73368841544607), 'PF': 1.5876204221860168, 'CVaR95': -4057.5945945945946}

[F] Active hybrid policy parameters:
{
  "tails": {
    "PUTS_0DTE_11": [
      0.06136325604608875,
      0.9607623915708898
    ],
    "CALLS_0DTE_11": [
      0.05136294962190549,
      0.9493190093797197
    ]
  },
  "gammas": {
    "PUTS_1DTE_1515": 0.0,
    "CALLS_1DTE_1515": 1.0
  }
}

[F] Hybrid portfolio summary: {'EDP': np.float64(173.52), 'PF': 1.88, 'CVaR95': -2894.76}


In [ ]:
# =========================================
# CELL G — Persist LIVE KIT for inference
# =========================================
import json, pickle, hashlib, numpy as np, pandas as pd
from lightgbm import LGBMRegressor

LIVE_DIR = OUT_DIR / "live_kit"
LIVE_DIR.mkdir(exist_ok=True, parents=True)

def _fit_full_model(book, feats, alpha):
    df = book_panel[book_panel['book']==book].copy().sort_values('open_date')
    X, y = df[feats], df["PNL"].fillna(0)
    m = LGBMRegressor(objective="quantile", alpha=alpha, n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
    # numeric-only safety (same coercion you used)
    for c in X.columns:
        if not (pd.api.types.is_numeric_dtype(X[c]) or pd.api.types.is_bool_dtype(X[c])): X[c]=pd.to_numeric(X[c],errors="coerce")
    m.fit(X, y)
    return m

def _train_decile_edges(scores, n=10):
    e = np.unique(np.nanpercentile(scores, np.linspace(0,100,n+1)))
    return e if len(e)>=3 else np.unique(np.nanpercentile(scores, np.linspace(0,100,6)))

def _cvar(s, alpha=0.05):
    v = pd.Series(s).dropna().values
    if v.size==0: return np.nan
    k=max(1,int(np.floor(alpha*len(v)))); idx=np.argsort(v)
    return float(np.mean(v[idx[:k]]))

kit = {
  "meta": {
    "train_end": str(book_panel["open_date"].max().date()),
    "features_by_book": {},
    "version": "xdte_pipeline_v1"
  },
  "puts": {},
  "calls": {},
  "hybrid_policy": {}
}

# 1) Save hybrid policy you already tuned
with open(OUT_DIR / "hybrid_policy_best.json") as f:
    kit["hybrid_policy"] = json.load(f)

# 2) Build per-book artifacts from LAST MONTHLY RUN objects you already computed
ALPHA_PUTS  = ALPHA
ALPHA_CALLS = ALPHA

for book in BOOKS:
    # use the same feature config you trained with
    feats = [f for f in BOOK_FEATS_CONFIG[book] if f in book_panel.columns]
    kit["meta"]["features_by_book"][book] = feats

    # fit a single full-model to use for live scoring
    m = _fit_full_model(book, feats, alpha=(ALPHA_PUTS if book.startswith("PUTS") else ALPHA_CALLS))
    # save model
    with open(LIVE_DIR / f"{book}_model.pkl", "wb") as f: pickle.dump(m, f)

    # derive train score series for edges
    df = book_panel[book_panel['book']==book].copy().sort_values('open_date')
    X = df[feats].copy()
    for c in X.columns:
        if not (pd.api.types.is_numeric_dtype(X[c]) or pd.api.types.is_bool_dtype(X[c])): X[c]=pd.to_numeric(X[c],errors="coerce")
    train_scores = pd.Series(m.predict(X), index=df.index)
    train_pnl    = df["PNL"].fillna(0).values

    if book.startswith("PUTS"):
        # store both τ and side chosen by TRAIN CVaR
        keep = float(json.load(open(OUT_DIR / "discovered_rules.json"))[book]["keep_target"])
        tau_top = np.nanquantile(train_scores.values, 1.0 - keep)
        tau_bot = np.nanquantile(train_scores.values, keep)
        c_top   = _cvar(df["PNL"].where(train_scores>=tau_top, 0.0))
        c_bot   = _cvar(df["PNL"].where(train_scores<=tau_bot, 0.0))
        use_top = (pd.notna(c_top) and pd.notna(c_bot) and (c_top >= c_bot)) or (pd.isna(c_bot) and pd.notna(c_top))
        kit["puts"][book] = {
            "keep": keep,
            "tau_top": float(tau_top),
            "tau_bot": float(tau_bot),
            "use_top": bool(use_top)
        }
    else:
        # store decile edges & whitelists from discovered rules
        rules = json.load(open(OUT_DIR / "discovered_rules.json"))
        edges = _train_decile_edges(train_scores.values, n=10).tolist()
        kit["calls"][book] = {
            "edges": edges,
            "long_deciles": rules[book]["long_deciles"],
            "short_deciles": rules[book]["short_deciles"]
        }

# 3) write live kit
with open(LIVE_DIR / "live_kit.json","w") as f: json.dump(kit, f, indent=2)
print(f"[LIVE KIT] wrote: {LIVE_DIR}")


[LIVE KIT] wrote: /content/drive/MyDrive/xdte_selector/backtest_data/_analysis_final_strategy/live_kit


In [ ]:
# =====================================
# MONTHLY RE-TUNE WRAPPER (safe update)
# =====================================
import json

CURRENT = json.load(open(OUT_DIR / "hybrid_policy_best.json"))

results = {"current": {}, "candidate": {}}

def evaluate_policy(policy):
    comp=[]
    for b in BOOKS:
        if b in policy.get("tails", {}):
            ql, qh = policy["tails"][b]; comp.append(_apply_tail_on_kill(b, ql, qh))
        elif b in policy.get("gammas", {}):
            comp.append(_apply_gamma_on_kill(b, float(policy["gammas"][b])))
        else:
            comp.append(final_daily_pnl[b].astype(float).reindex(DAYS))
    port = pd.concat(comp, axis=1).sum(axis=1)
    return {"EDP": port.mean(), "PF": _pf(port), "CVaR95": _cvar(port)}

# 1) Evaluate current
results["current"] = evaluate_policy(CURRENT)

# 2) Quick local sweep around current (±0.01 on tails; gammas from {1,.75,.5})
def clamp(x,a,b): return max(a, min(b, x))
cands = []
for b, (ql, qh) in CURRENT["tails"].items():
    for dql in [-0.01, 0, +0.01]:
        for dqh in [-0.01, 0, +0.01]:
            if dql==0 and dqh==0: continue
            cand = {"tails": CURRENT["tails"].copy(), "gammas": CURRENT["gammas"].copy()}
            ql2 = clamp(ql + dql, 0.01, 0.10); qh2 = clamp(qh + dqh, 0.90, 0.99)
            if ql2 >= qh2: continue
            cand["tails"] = cand["tails"].copy(); cand["tails"][b] = [ql2, qh2]
            cands.append(cand)

# PUTS_1DTE_1515 gamma try (stays at 1.0 by default)
for g in [1.0, 0.75, 0.5]:
    cand = {"tails": CURRENT["tails"].copy(), "gammas": CURRENT["gammas"].copy()}
    cand["gammas"]["PUTS_1DTE_1515"] = g
    cands.append(cand)

best = {"PF": -1, "CVaR95": -9e9, "EDP": -1, "policy": CURRENT}
for cand in cands:
    m = evaluate_policy(cand)
    # constraints vs no-kill
    edp_floor = 0.95 * final_daily_pnl[BOOKS].sum(axis=1).mean()
    if m["EDP"] < edp_floor:
        continue
    # keep only candidates with PF>=current PF and CVaR>=current CVaR
    if (m["PF"] >= results["current"]["PF"]) and (m["CVaR95"] >= results["current"]["CVaR95"]):
        if (m["PF"] > best["PF"]) or (m["PF"]==best["PF"] and m["CVaR95"]>best["CVaR95"]):
            best.update({**m, "policy": cand})

results["candidate"] = {k: best[k] for k in ["EDP","PF","CVaR95"]}
print("\n=== Re-tune result (local sweep) ===")
print("Current:", results["current"])
print("Best   :", results["candidate"])

# 3) Write only if strictly better on PF and CVaR (EDP meets floor)
if (results["candidate"]["PF"]  > results["current"]["PF"]) and \
   (results["candidate"]["CVaR95"] > results["current"]["CVaR95"]):
    with open(OUT_DIR / "hybrid_policy_best.json", "w") as f:
        json.dump(best["policy"], f, indent=2)
    print("[UPDATE] Policy file updated.")
else:
    print("[KEEP] Existing policy retained.")



=== Re-tune result (local sweep) ===
Current: {'EDP': np.float64(173.5179760319574), 'PF': 1.877421439969835, 'CVaR95': -2894.7567567567567}
Best   : {'EDP': np.float64(173.6977363515313), 'PF': 1.8791295440147726, 'CVaR95': -2894.7567567567567}
[KEEP] Existing policy retained.
